# Exercise 5.1 — Overfitting and underfitting

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-01-overfitting.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

A model is only useful if it predicts **people it has not seen**. So we always
compare two errors: on the training rows, which the model was fitted on, and on
the test rows, which it has never seen.

- **Underfitting**: the model is too simple. The error is large on both.
- **Overfitting**: the model is too flexible. It also learns the noise in the
  training rows, so the training error is small but the test error is larger.

We will see both with one predictor and three models: a straight line, a
gentle curve and a very bendy one.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

The data are in [`data/drug_levels.csv`](data/drug_levels.csv): a small pilot
study of 20 patients. Each patient had one blood sample taken between 0 and 3
hours after a dose of a drug. `hours` is the time since the dose and
`concentration` the drug level in the blood (mg/L). As always, 80% of the
patients are used for training and 20% are held back as the test set.

In [ ]:
drugs = pd.read_csv("data/drug_levels.csv")

train, test = train_test_split(drugs, test_size=0.2, random_state=42)

print(len(train), len(test))

16 patients for training and 4 for testing. A small study, which is exactly
where overfitting is easiest to fall into.

------------------------------------------------------------------------

### Part 1 — Look at the data

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.scatter(train["hours"], train["concentration"], color="black", s=20)
ax.scatter(test["hours"], test["concentration"], color="red", s=40)
ax.set_xlabel("Hours since the dose")
ax.set_ylabel("Concentration (mg/L)")
ax.set_title("Training patients (black) and test patients (red)")
plt.show()

**Question!** Would a straight line describe these points well?

The level rises for about an hour and then falls. A straight line cannot do
that.

------------------------------------------------------------------------

### Part 2 — Three models

All three are linear regressions of `concentration` on `hours`. Only the first
step of the pipeline changes. `PolynomialFeatures(degree=3, include_bias=False)`
turns the one column `hours` into three columns, $\text{hours}$,
$\text{hours}^2$ and $\text{hours}^3$, and the extra columns let the fitted line
bend. The higher the degree, the more the line can bend.

See the three columns it makes for the first three training patients:

In [ ]:
X_train, y_train = train[["hours"]], train["concentration"]
X_test,  y_test  = test[["hours"]],  test["concentration"]

PolynomialFeatures(degree=3, include_bias=False).fit_transform(X_train)[:3].round(2)

Now fit the three models on the training rows: a straight line (degree 1), a
gentle curve (degree 3) and a very bendy curve (degree 6).

In [ ]:
# Replace every ______ below, then run this cell.
fit_1 = Pipeline([("poly", PolynomialFeatures(degree=1, include_bias=False)),
                  ("linreg", LinearRegression())])
fit_1.fit(X_train, y_train)

fit_3 = Pipeline([("poly", PolynomialFeatures(degree=3, include_bias=False)),
                  ("linreg", LinearRegression())])
fit_3.fit(X_train, y_train)

fit_6 = Pipeline([("poly", PolynomialFeatures(degree=______, include_bias=False)),
                  ("linreg", LinearRegression())])
fit_6.fit(______, y_train)

> **Stuck?**
>
> The bendy curve has degree 6, and every model is fitted on the training rows
> only, so `X_train`.

------------------------------------------------------------------------

### Part 3 — Training error and test error

For each model we predict the training rows and the test rows, and measure the
error with the RMSE: the square root of the mean squared error from Day 3, in
the units of the outcome (mg/L). Here is the straight line:

In [ ]:
train_rmse_1 = root_mean_squared_error(y_train, fit_1.predict(X_train))
test_rmse_1  = root_mean_squared_error(y_test,  fit_1.predict(X_test))

Do the same for the other two models:

In [ ]:
# Replace every ______ below, then run this cell.
train_rmse_3 = root_mean_squared_error(y_train, fit_3.predict(X_train))
test_rmse_3  = root_mean_squared_error(______, fit_3.predict(______))

train_rmse_6 = root_mean_squared_error(y_train, fit_6.predict(X_train))
test_rmse_6  = root_mean_squared_error(______, fit_6.predict(______))

> **Stuck?**
>
> Copy the straight-line cell: the test RMSE compares `y_test` with the
> predictions for `X_test`.

------------------------------------------------------------------------

### Part 4 — Compare the three models

All the results in one table:

In [ ]:
results = pd.DataFrame({
    "model":      ["degree 1", "degree 3", "degree 6"],
    "train_rmse": [train_rmse_1, train_rmse_3, train_rmse_6],
    "test_rmse":  [test_rmse_1,  test_rmse_3,  test_rmse_6],
})
print(results.round(2))

And the three fitted curves:

In [ ]:
hours_grid = pd.DataFrame({"hours": np.linspace(drugs["hours"].min(), drugs["hours"].max(), 300)})
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5), sharey=True)
for ax, model, name in zip(axes, [fit_1, fit_3, fit_6], ["degree 1", "degree 3", "degree 6"]):
    ax.plot(hours_grid["hours"], model.predict(hours_grid), color="steelblue", linewidth=2)
    ax.scatter(train["hours"], train["concentration"], color="black", s=20)
    ax.scatter(test["hours"], test["concentration"], color="red", s=40)
    ax.set_title(name)
    ax.set_xlabel("Hours since the dose")
axes[0].set_ylabel("Concentration (mg/L)")
fig.suptitle("Three fitted curves; training patients black, test patients red")
plt.tight_layout()
plt.show()

**Question!** Which model underfits, which overfits, and which would you use?

| Model | Training RMSE | Test RMSE | |
|:-|:-:|:-:|:-|
| degree 1 | 2.15 | 1.28 | underfits: large on both |
| degree 3 | 1.04 | 0.65 | fits: small on both |
| degree 6 | 0.63 | 1.37 | overfits: smallest on training, larger on test |

The straight line misses the rise and fall, so its error is large on both sets.
The degree-3 curve follows the shape. The degree-6 curve shoots up to about
16 mg/L before the first hour, to pass through the one high training patient:
it has the smallest training error of the three, but on the test patients it is
worse than the degree-3 curve. A smaller training error does not mean a better
model. Only the test rows can tell.

> **Key points**
>
> - Always compare the error on the training rows with the error on the test rows.
> - Underfitting: large error on both. The model is too simple.
> - Overfitting: small error on training, larger on test. The model has learnt the noise.
> - More flexibility always lowers the training error, so the training error cannot choose a model.